<a href="https://colab.research.google.com/github/mirnaelsheikhh/NLP-Fake-News-Detection/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install nltk pandas -q

import os
import re
import ast
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

nltk.download('punkt', quiet=True)
nltk.download('wordnet',quiet=True)
nltk.download('stopwords',quiet=True)
nltk.download('omw-1.4',quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('punkt_tab', quiet=True)

wn          = nltk.WordNetLemmatizer()
stopwordsEn = set(stopwords.words('english'))

CLEAN_DATA_PATH = "/content/drive/MyDrive/dataset_english_only.csv"
PREPROCESSED_PATH = "/content/drive/MyDrive/dataset_preprocessed.csv"
MODEL_DIR   = "/content/drive/MyDrive/fake_news_models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Setup complete!")

Mounted at /content/drive
Setup complete!


In [ ]:
print("Loading dataset from Drive...")
df = pd.read_csv(CLEAN_DATA_PATH)
print("Shape:", df.shape)
print(df.head(2))

Loading dataset from Drive...
Shape: (58417, 6)
                                               title  \
0  WARNING: A Pivotal Moment For The Stock Market...   
1  Trump, top defense officials, discuss North Ko...   

                                                text  unreliable  \
0  WARNING: A Pivotal Moment For The Stock Market...           1   
1  WASHINGTON  - U.S. President Donald Trump met ...           0   

                            author  word_count  char_length  
0  Anonymous Coward (UID 72071746)          43          228  
1                          Unknown          85          426  


In [ ]:
def removePunctuation(text):
    return re.sub(r"[^\w\s]", "", text)

def lowerCaseText(text):
    return text.lower()

def tokenize(text):
    return word_tokenize(text)

def removeStopWords(tokens):
    return [word for word in tokens if word not in stopwordsEn]

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def getWordnetPOS(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('N'): return wordnet.NOUN
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def lemmetization(tokens):
    pos_tags = nltk.pos_tag(tokens)
    return [wn.lemmatize(word, getWordnetPOS(tag)) for word, tag in pos_tags]

def preprocessingText2(text):
    if not isinstance(text, str):
        return []
    text = lowerCaseText(text)
    text = remove_numbers(text)
    text = removePunctuation(text)
    text = tokenize(text)
    text = removeStopWords(text)
    text = lemmetization(text)
    return text

def parse_tokens(val):
    if isinstance(val, list): return val
    if isinstance(val, str):
        try: return ast.literal_eval(val)
        except: return val.split()
    return []

print("Functions defined")

Functions defined


In [ ]:
if os.path.exists(PREPROCESSED_PATH):
    print("Loading existing preprocessed dataset")
    df = pd.read_csv(PREPROCESSED_PATH)
    df["preprocessingText2"] = df["preprocessingText2"].apply(parse_tokens)
else:
    print("Preprocessing text")
    df["preprocessingText2"] = df["text"].apply(preprocessingText2)
    df["cleanText"] = df["preprocessingText2"].apply(
        lambda x: " ".join(x) if isinstance(x, list) else ""
    )
    df.to_csv(PREPROCESSED_PATH, index=False)
    print("Saved preprocessed dataset to Drive")

df["cleanText"] = df["preprocessingText2"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)
print("Sample:")
print(df[["preprocessingText2", "cleanText"]].head(2))

Preprocessing text
Saved preprocessed dataset to Drive
Sample:
                                  preprocessingText2  \
0  [warn, pivotal, moment, stock, market, may, pa...   
1  [washington, u, president, donald, trump, meet...   

                                           cleanText  
0  warn pivotal moment stock market may page mail...  
1  washington u president donald trump meet tuesd...  


In [ ]:
# Vocabulary
vocab = set()
for row in df["preprocessingText2"]:
    vocab.update(row)
print("Vocabulary size:", len(vocab))


Vocabulary size: 286578


In [ ]:
# Class word counters
fakeWords, realWords = [], []
for _, row in df.iterrows():
    if row["unreliable"] == 0:
        fakeWords.extend(row["preprocessingText2"])
    else:
        realWords.extend(row["preprocessingText2"])

fakeCounter = Counter(fakeWords)
realCounter = Counter(realWords)

print("\nFAKE TOP WORDS:", fakeCounter.most_common(10))
print("REAL TOP WORDS:", realCounter.most_common(10))




FAKE TOP WORDS: [('say', 195103), ('trump', 86078), ('mr', 67272), ('state', 59446), ('would', 53937), ('u', 51191), ('president', 43932), ('year', 40627), ('new', 36431), ('one', 36371)]
REAL TOP WORDS: [('trump', 82722), ('say', 59110), ('people', 37320), ('u', 34877), ('one', 34379), ('state', 32763), ('clinton', 32471), ('would', 32416), ('go', 31575), ('make', 30684)]


In [ ]:
# Fake patterns
patterns = [
    (word, freq) for word, freq in fakeCounter.items()
    if freq > realCounter.get(word, 0)
]
patterns.sort(key=lambda x: x[1], reverse=True)
print("\nFAKE PATTERNS (top 20):", patterns[:20])


FAKE PATTERNS (top 20): [('say', 195103), ('trump', 86078), ('mr', 67272), ('state', 59446), ('would', 53937), ('u', 51191), ('president', 43932), ('year', 40627), ('new', 36431), ('one', 36371), ('make', 33062), ('also', 31275), ('government', 28361), ('take', 28329), ('republican', 27564), ('time', 27200), ('house', 25710), ('could', 25418), ('last', 23861), ('tell', 23370)]
